In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.preprocessing import LabelEncoder

In [2]:
csv_path = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv'
img_dir = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train'

df_train = pd.read_csv(csv_path)
label_encoder = LabelEncoder()
target_col = df_train.columns[1] # คอลัมน์ 'class'
df_train['class_index'] = label_encoder.fit_transform(df_train[target_col])

In [4]:
test_csv_path = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv'
test_df = pd.read_csv(test_csv_path)

In [5]:
class HouseTrainDataset(Dataset): 
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # คลีนชื่อไฟล์ (วิธีนี้กันเหนียวได้ชัวร์)
        img_id_str = str(self.df.iloc[idx, 0]).strip()
        img_name = img_id_str.replace('.jpg', '') + '.jpg'
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.iloc[idx]['class_index']) # ดึงเฉลยมาเป็นตัวเลข
        
        if self.transform:
            image = self.transform(image)
        return image, label

อันนี้ตอนใช้ mobilenetv2

In [ ]:
# ปรับตัวแปร transform ใหม่ (ใช้ตอนเทรน)
transform = transforms.Compose([
    transforms.Resize((256, 256)),      # ขยายใหญ่ขึ้นนิดนึงก่อน Crop
    transforms.RandomResizedCrop(224), # สุ่มซูมเข้าไปในภาพ (ช่วยเรื่องบ้านอยู่ไกลหรือโดนบัง)
    transforms.RandomHorizontalFlip(),  # พลิกซ้ายขวา (บ้านหันคนละทาง)
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # สู้กับภาพที่มืดหรือสว่างเกินไป
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ค่ามาตรฐานของ ImageNet
])
# ประกอบร่างเข้าด้วยกัน
train_dataset = HouseTrainDataset(df_train, img_dir, transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

print(f"✅ เตรียม DataLoader สำเร็จ! พร้อมเข้าสู่ลูป Training แล้วครับ")

ของ dinov2 

In [6]:
transform = transforms.Compose([
    # เปลี่ยนจาก 224 เป็น 518 ให้ตรงกับความต้องการของ DINOv2
    transforms.Resize((518, 518)), 
    transforms.ToTensor(),
    # ใช้ค่า Normalize มาตรฐานของ ImageNet (DINOv2 ใช้ค่านี้ครับ)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
# ประกอบร่างเข้าด้วยกัน
train_dataset = HouseTrainDataset(df_train, img_dir, transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"✅ เตรียม DataLoader สำเร็จ! พร้อมเข้าสู่ลูป Training แล้วครับ")

✅ เตรียม DataLoader สำเร็จ! พร้อมเข้าสู่ลูป Training แล้วครับ


In [ ]:
# 4. สร้าง DataLoader สำหรับส่งเข้าโมเดล
test_dir = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test/'
test_dataset = KaggleTestDataset(dataframe=test_df, img_dir=test_dir, transform=transform)

# **จุดสำคัญมาก:** สำหรับชุด Test ห้ามสลับข้อเด็ดขาด (shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("เตรียมข้อมูล Test สำหรับนำไปทำนายผลเรียบร้อยแล้ว!")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# 2. ปรับสมองส่วนปลายให้ทายแค่ 2 คลาสเหมือนเดิม
model.classifier[1] = nn.Linear(model.last_channel, len(label_encoder.classes_))
model = model.to(device)

print("🚀 โหลดโมเดลพร้อมสมอง Transfer Learning เรียบร้อย!")

In [7]:
import timm
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 1. โหลด DINOv2 โดยระบุ num_classes ตั้งแต่ตอนสร้างเลย!
model_name = 'vit_small_patch14_dinov2.lvd142m'
print(f"กำลังประกอบร่าง {model_name}...")

# ใส่ num_classes เข้าไปในฟังก์ชันสร้างเลย timm จะจัดการหัวให้เอง
model = timm.create_model(
    model_name, 
    pretrained=True, 
    num_classes=len(label_encoder.classes_)
)

# 2. เช็คความถูกต้อง (Optional)
print(f"✅ ส่วนหัวถูกเปลี่ยนเป็น: {model.get_classifier()}")

model = model.to(device)
print("🚀 DINOv2 พร้อมออกรบแล้วครับชูจิ!")

# 3. ตั้งค่า Optimizer (สำคัญ: ViT ต้องใช้ Learning Rate ต่ำๆ นะครับ)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()

กำลังประกอบร่าง vit_small_patch14_dinov2.lvd142m...
✅ ส่วนหัวถูกเปลี่ยนเป็น: Linear(in_features=384, out_features=2, bias=True)
🚀 DINOv2 พร้อมออกรบแล้วครับชูจิ!


In [ ]:
# 1. โหลดข้อมูล
df = pd.read_csv('train.csv')

# 2. ตรวจสอบชื่อคอลัมน์ (สมมติว่าเรารู้อยู่แล้วว่าคอลัมน์แรกคือ ID คอลัมน์สองคือคำตอบ)
# เราสามารถอ้างอิงด้วย "ลำดับของคอลัมน์" แทน "ชื่อ" ได้ครับ
target_column = df.columns[1] # หยิบชื่อคอลัมน์ลำดับที่ 2 ออกมา (index 1)
print(f"กำลังใช้คอลัมน์ '{target_column}' เป็นตัวแปรเป้าหมาย")

# 3. แปลงรหัส
label_encoder = LabelEncoder()
df['class_index'] = label_encoder.fit_transform(df[target_column])

num_classes = len(label_encoder.classes_)
print(f"เตรียมข้อมูลเสร็จ! พบทั้งหมด {num_classes} คลาส")

In [ ]:
# 1. กำหนดอุปกรณ์ (แก้คำหล่นด้านหลังเป็น "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- โค้ดเสริม: เช็คให้ชัวร์ว่า P100 มาจริงไหม! ---
if torch.cuda.is_available():
    print(f"🔥 สุดยอด! ตอนนี้กำลังรันบนการ์ดจอ: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ คำเตือน: ตอนนี้รันบน CPU ธรรมดา (อย่าลืมไปเปิด GPU ในเมนู Kaggle นะครับ)")

# 2. ประกาศโครงสร้างสมอง (Architecture) ให้เหมือนตอน Train
model = models.mobilenet_v2(weights=None)
num_classes = 2  
model.classifier[1] = nn.Linear(model.last_channel, num_classes)

# 3. ส่งโมเดลไปหา P100
model = model.to(device)

print("✅ ประกาศตัวแปร model และโหลดโครงสร้างลงอุปกรณ์เรียบร้อยแล้ว!")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()
epochs = 15 

print("🚀 โครงสร้างสมบูรณ์! เริ่มต้นการเทรน...")
model.train()
for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader: 
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()               
        outputs = model(images)             
        loss = criterion(outputs, labels)   
        loss.backward()                     
        optimizer.step()                    
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Acc: {accuracy:.2f}%")

In [9]:
from torch.cuda.amp import GradScaler, autocast
# --- 2. ตั้งค่า Gradient Accumulation และ AMP ---
accumulation_steps = 16  # เดิน 16 รอบ (2x16 = 32) เพื่อให้ได้ผลเท่ากับ Batch 32
scaler = GradScaler()     # ตัวช่วยสำหรับ Mixed Precision
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()
epochs = 15 

model.train()
optimizer.zero_grad()

for epoch in range(epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # --- ท่าที่ 1: Mixed Precision (คำนวณแบบประหยัดแรม) ---
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss = loss / accumulation_steps # หารเฉลี่ยตามจำนวนก้าวที่สะสม

        # --- ท่าที่ 2: Scaled Backward ---
        scaler.scale(loss).backward()

        # --- ท่าที่ 3: Gradient Accumulation (สะสมครบก้าวค่อยก้าวเดิน) ---
        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
            # เคลียร์เศษขยะในแรมการ์ดจอเป็นระยะ
            torch.cuda.empty_cache()

    print(f"Epoch {epoch+1} เสร็จสิ้น!")

/tmp/ipykernel_514/2064833519.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()     # ตัวช่วยสำหรับ Mixed Precision
/tmp/ipykernel_514/2064833519.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 เสร็จสิ้น!
Epoch 2 เสร็จสิ้น!
Epoch 3 เสร็จสิ้น!
Epoch 4 เสร็จสิ้น!
Epoch 5 เสร็จสิ้น!
Epoch 6 เสร็จสิ้น!
Epoch 7 เสร็จสิ้น!
Epoch 8 เสร็จสิ้น!
Epoch 9 เสร็จสิ้น!
Epoch 10 เสร็จสิ้น!
Epoch 11 เสร็จสิ้น!
Epoch 12 เสร็จสิ้น!
Epoch 13 เสร็จสิ้น!
Epoch 14 เสร็จสิ้น!
Epoch 15 เสร็จสิ้น!


In [10]:
class HouseTestDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # ดึง ID และจัดการนามสกุลไฟล์
        img_id_str = str(self.df.iloc[idx, 0]).strip()
        img_name = img_id_str.replace('.jpg', '') + '.jpg'
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        return image, img_id_str # ส่งรูปกับ ID กลับไป

In [11]:
test_csv_path = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv'
test_img_dir = '/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test'

df_test = pd.read_csv(test_csv_path)

# ใช้ transform เดียวกับตอน Train
test_dataset = HouseTestDataset(df_test, test_img_dir, transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # ห้าม Shuffle!

print(f"✅ สร้าง test_loader สำเร็จ! พร้อมทำนาย {len(test_dataset)} ข้อ")

# ==========================================
# 3. เริ่มทำนายและสร้างไฟล์ Submission
# ==========================================
predicted_indices = []
model.eval()

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        predicted_indices.extend(preds.cpu().numpy())

# แปลงเลขกลับเป็นรหัสบ้าน
predicted_answers = label_encoder.inverse_transform(predicted_indices)

# ใส่คำตอบลงตาราง
df_test['answer'] = predicted_answers
df_test.to_csv('submission_finaldino.csv', index=False)

print("🎉 สร้างไฟล์ submission_final.csv เรียบร้อยแล้วครับ ชูจิส่งได้เลย!")

✅ สร้าง test_loader สำเร็จ! พร้อมทำนาย 1550 ข้อ
🎉 สร้างไฟล์ submission_final.csv เรียบร้อยแล้วครับ ชูจิส่งได้เลย!


eda

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. นับจำนวนในแต่ละคลาส
class_counts = df_train['class'].value_counts()
class_perc = df_train['class'].value_counts(normalize=True) * 100

print("--- สรุปจำนวนข้อมูลในแต่ละคลาส ---")
for label, count in class_counts.items():
    print(f"คลาส {label}: {count} รูป ({class_perc[label]:.2f}%)")

# 2. วาดกราฟแท่งให้เห็นภาพชัดๆ
plt.figure(figsize=(8, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Data Distribution per Class')
plt.xlabel('House Class')
plt.ylabel('Number of Images')
plt.show()

imbalance data 

In [ ]:
# 1. แยกข้อมูลออกเป็น 2 กลุ่มตามคลาส
df_class_0 = df_train[df_train['class_index'] == 0]
df_class_1 = df_train[df_train['class_index'] == 1]

# 2. ดูว่าคลาส 1 (ที่มีน้อยกว่า) มีจำนวนเท่าไหร่
n_samples = len(df_class_1)

# 3. สุ่มหยิบคลาส 0 ออกมาให้เท่ากับจำนวนคลาส 1 เป๊ะๆ
df_class_0_downsampled = df_class_0.sample(n_samples, random_state=42)

# 4. รวมร่างกลับเป็น df_train ชุดใหม่ที่สมดุล 50/50
df_train_balanced = pd.concat([df_class_0_downsampled, df_class_1])

# สลับลำดับแถวใหม่เพื่อไม่ให้โมเดลจำจำเจ
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"--- หลังปรับสมดุล ---")
print(f"จำนวนคลาส 0: {len(df_train_balanced[df_train_balanced['class_index'] == 0])} รูป")
print(f"จำนวนคลาส 1: {len(df_train_balanced[df_train_balanced['class_index'] == 1])} รูป")

# 5. อัปเดต train_dataset และ train_loader ใหม่
train_dataset = HouseTrainDataset(df_train_balanced, img_dir, transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_samples(df, img_dir, n_samples=5):
    classes = df['class'].unique()
    fig, axes = plt.subplots(len(classes), n_samples, figsize=(15, 6))
    
    for i, cls in enumerate(classes):
        # สุ่มภาพจากแต่ละคลาส
        sample_df = df[df['class'] == cls].sample(n_samples, random_state=42)
        
        for j, (_, row) in enumerate(sample_df.iterrows()):
            img_name = str(row[0]).strip().replace('.jpg', '') + '.jpg'
            img_path = os.path.join(img_dir, img_name)
            
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].set_axis_off()
            if j == 0:
                axes[i, j].set_title(f"Class: {cls}", loc='left', fontweight='bold')

    plt.tight_layout()
    plt.show()

# เรียกใช้งาน (ใช้ df_train_balanced ที่เราเพิ่งทำก็ได้ครับ)
visualize_samples(df_train, img_dir)